# Consensus scoring — Boltz + ESMFold + Protenix

Scores designed binders against deGFP with **three independent predictors** and ranks by
agreement rather than by any single metric.

---

## Why agreement, not ipSAE

From the DWV wet-lab results (20 designs ordered, 1 bound at 603 nM):

| metric | where the one real binder ranked |
|---|---|
| **ipSAE_min** | **10th of 20** |
| binding_confidence | 5th of 20 |
| Protenix ipTM | **1st of 17** |
| ESMFold ipTM | **2nd of 17** |
| Protenix seed spread | **1st of 17** |

Rank-sum across all four put it **1st, by a 7-point margin**. A hard filter on all four passed
exactly one design out of twenty - the binder.

**Caveat, stated plainly: n = 1.** One positive, twenty designs, five metrics. The chance a random
design lands top-2 on at least one of five metrics is ~47%. What is harder to dismiss is that it
was top-2 on *three at once* (~1.4% by chance). Treat this as a **prior worth acting on, not a
validated filter.** This notebook is how you test it prospectively.

## What each predictor is actually measuring

* **Boltz** (MSA + structure priors) -> ipSAE, and seed spread across samples. *Is the interface right,
  and does the model agree with itself?*
* **ESMFold** (language model, no MSA, no template) -> folds from sequence statistics alone.
  On the DWV data this was the only metric that tracked **expression** (Spearman +0.34 with
  whether the protein loaded; Protenix was -0.10). *Does this look like a real protein?*
* **Protenix** -> independent third opinion on the interface.

---

## Crash-proofing

* Every result is appended to **one JSON checkpoint on Drive**, keyed by `(predictor, design)`.
* Re-running any cell **skips work already done**. A disconnect costs you one design, not the run.
* **Run ONE predictor per session.** ESMFold, Protenix and Boltz have conflicting dependency
  stacks. Set `PREDICTOR` below, run, then `Runtime > Restart`, change it, run again. The
  checkpoint accumulates across all of them.


## 0 · Drive guard — run this FIRST, every session

If Drive is not actually writable, everything below lands on the VM's ephemeral disk and is
**gone** when the runtime recycles. This cell refuses to let that happen silently.

In [ ]:
import os, sys, time, json
from google.colab import drive

MOUNT = "/content/drive"
if not os.path.isdir(f"{MOUNT}/MyDrive"):
    drive.mount(MOUNT, force_remount=False)

# Hard check: mounted AND writable AND the write survives a re-read.
probe = f"{MOUNT}/MyDrive/.consensus_write_probe"
try:
    stamp = str(time.time())
    open(probe, "w").write(stamp)
    assert open(probe).read() == stamp
    os.remove(probe)
except Exception as e:
    raise SystemExit(
        f"FATAL: Google Drive is not writable ({e}).\n"
        "Everything this notebook produces would be lost on disconnect.\n"
        "Fix: Runtime > Disconnect and delete runtime, reconnect, re-run this cell,\n"
        "and accept the Drive authorisation prompt when it appears.")

FREE_GB = os.statvfs(f"{MOUNT}/MyDrive").f_bavail * os.statvfs(f"{MOUNT}/MyDrive").f_frsize / 1e9
print(f"Drive mounted and writable. ~{FREE_GB:.1f} GB free.")
if FREE_GB < 5:
    print("!! WARNING: under 5 GB free. Protenix and Boltz write a lot of structures. Clear space first.")

print("\nNOTE on free Colab: ~90 min idle timeout and a 12 h hard cap. Drive persistence means")
print("you never LOSE work, but the job still STOPS. Re-run the launch cell to continue.")
print("Keep the browser tab open and visible to reduce idle disconnects.")


## 1 · Config and checkpoint

In [ ]:
import os, json, glob, hashlib, time, sys
from google.colab import drive
drive.mount('/content/drive')

# ----------------------------- CONFIG -----------------------------
PREDICTOR = "esmfold"        # "esmfold" | "protenix" | "boltz"   <-- ONE per session
DRIVE_ROOT = "/content/drive/MyDrive/BindCraft"
BINDERS_FA = f"{DRIVE_ROOT}/deGFP_M_core_hs151-166-180-200_binders.fasta"   # from notebook 1 cell 8
CKPT       = f"{DRIVE_ROOT}/consensus_scores.json"
N_SEED     = 3               # samples per design, for seed spread
# ------------------------------------------------------------------

# deGFP target, 4EUL chain A 6-229, chromophore gap at 64->68 (see notebook 1)
TARGET_SEQ = ("KGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTL"
              "TYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELK"
              "GIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPI"
              "GDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK")

os.makedirs(DRIVE_ROOT, exist_ok=True)
ckpt = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
def key(p, n): return f"{p}::{n}"
def save():
    tmp = CKPT + ".tmp"
    json.dump(ckpt, open(tmp, "w"), indent=1); os.replace(tmp, CKPT)   # atomic, survives a kill

designs = {}
if os.path.exists(BINDERS_FA):
    name = None
    for line in open(BINDERS_FA):
        line = line.strip()
        if line.startswith(">"): name = line[1:].split()[0]; designs[name] = ""
        elif name: designs[name] += line
else:
    print(f"!! {BINDERS_FA} not found. Either run notebook 1 cell 8, or paste sequences below.")
    designs = {}   # e.g. {"seed16": "SGALKYGET...", "seed12": "SDAEWELAA..."}

todo = [n for n in designs if key(PREDICTOR, n) not in ckpt]
print(f"predictor : {PREDICTOR}")
print(f"designs   : {len(designs)}   already scored: {len(designs)-len(todo)}   to do: {len(todo)}")
print(f"checkpoint: {CKPT}")


## 2 · ESMFold

Language model, no MSA, no template. Folds from sequence statistics alone — which is exactly why
it behaves as a **naturalness / expressibility** signal rather than an interface signal.

Complexes are done with a **25-residue glycine linker**, the standard ESMFold multimer hack.
That makes the interface number a rough proxy; the monomer pLDDT is the cleaner read on
"will this protein exist."

Needs ~16 GB VRAM in fp16. Fine on A100 or L4. On a T4 it is tight — if you OOM, score
monomers only by setting `COMPLEX = False`.

In [ ]:
if PREDICTOR == "esmfold":
    !pip install -q transformers accelerate
    import torch
    from transformers import AutoTokenizer, EsmForProteinFolding

    COMPLEX = True
    tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
    model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
    model = model.cuda().eval()
    model.esm = model.esm.half()
    torch.backends.cuda.matmul.allow_tf32 = True

    def fold(seq):
        with torch.no_grad():
            enc = tok([seq], return_tensors="pt", add_special_tokens=False)
            out = model(enc["input_ids"].cuda())
        plddt = out["plddt"][0, :, 1].mean().item()          # CA pLDDT, 0-1
        ptm   = out["ptm"].item() if "ptm" in out else float("nan")
        return plddt, ptm

    for i, n in enumerate(todo, 1):
        s = designs[n]
        try:
            mono_plddt, mono_ptm = fold(s)
            rec = {"esm_monomer_plddt": mono_plddt, "esm_monomer_ptm": mono_ptm, "len": len(s)}
            if COMPLEX:
                cx_plddt, cx_ptm = fold(TARGET_SEQ + "G"*25 + s)
                rec.update({"esm_complex_plddt": cx_plddt, "esm_complex_ptm": cx_ptm})
            ckpt[key("esmfold", n)] = rec; save()
            print(f"[{i}/{len(todo)}] {n:<28} mono pLDDT {mono_plddt:.3f}"
                  + (f"  complex pTM {cx_ptm:.3f}" if COMPLEX else ""))
        except RuntimeError as e:
            print(f"[{i}/{len(todo)}] {n} FAILED: {e}")
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                print("  OOM - set COMPLEX=False, or switch to an A100/L4 runtime")
    print("done. Runtime > Restart, then set PREDICTOR to the next one.")
else:
    print(f"skipped (PREDICTOR={PREDICTOR})")


## 3 · Protenix

Third independent opinion on the interface, plus **seed spread** — re-predict the same complex with
different seeds and measure how much the answer moves. Low spread means the model lands on one
pose every time; high spread means it is guessing differently each run and you only saw one answer.

On the DWV panel, seed spread by itself predicted **neither** binding nor expression
(Spearman +0.10 with loading). It is worth computing as a **sanity gate** — throw out designs where
the model contradicts itself — not as a ranking metric.

**Verify the CLI before trusting this cell.** Protenix's interface changes between releases; the
first line prints `--help` so you can check the flags against what is written here.

In [ ]:
if PREDICTOR == "protenix":
    !pip install -q protenix
    !protenix predict --help 2>&1 | head -40      # CHECK THESE FLAGS before relying on the loop

    import subprocess, numpy as np, tempfile
    OUT = "/content/protenix_out"; os.makedirs(OUT, exist_ok=True)

    def run_protenix(name, seq, seed):
        d = f"{OUT}/{name}_s{seed}"
        if os.path.exists(f"{d}/done.txt"):
            return d
        os.makedirs(d, exist_ok=True)
        job = {"name": name, "sequences": [
            {"proteinChain": {"sequence": TARGET_SEQ, "count": 1}},
            {"proteinChain": {"sequence": seq,        "count": 1}}]}
        jp = f"{d}/input.json"; json.dump([job], open(jp, "w"))
        r = subprocess.run(["protenix","predict","--input",jp,"--out_dir",d,
                            "--seeds",str(seed),"--use_msa_server"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f"    protenix rc={r.returncode}: {r.stderr[-400:]}"); return None
        open(f"{d}/done.txt","w").write("ok"); return d

    def read_iptm(d):
        for f in glob.glob(f"{d}/**/*summary_confidence*.json", recursive=True):
            j = json.load(open(f))
            v = j.get("iptm", j.get("ipTM"))
            if v is not None: return float(v)
        return None

    for i, n in enumerate(todo, 1):
        vals = []
        for seed in range(N_SEED):
            d = run_protenix(n, designs[n], seed)
            if d:
                v = read_iptm(d)
                if v is not None: vals.append(v)
        if vals:
            ckpt[key("protenix", n)] = {"protenix_iptm": float(np.mean(vals)),
                                        "protenix_seed_spread": float(np.max(vals)-np.min(vals)),
                                        "protenix_n_seeds": len(vals)}
            save()
            print(f"[{i}/{len(todo)}] {n:<28} ipTM {np.mean(vals):.3f}  spread {np.max(vals)-np.min(vals):.3f}")
        else:
            print(f"[{i}/{len(todo)}] {n} produced no usable output")
    print("done. Runtime > Restart, then set PREDICTOR to the next one.")
else:
    print(f"skipped (PREDICTOR={PREDICTOR})")


## 4 · Boltz — via the hosted API

Uses the `boltz-api` CLI you already have credits on. `num_samples` gives you replicate ipSAE
in one job, which is what the DWV analysis used (spreads of 0.007–0.016 on the deGFP finalists).

**This costs real money.** The cell runs `estimate-cost` and then **stops**. Read the figure,
then set `CONFIRMED = True` and re-run to submit. Nothing is billed until you do that.

In [ ]:
if PREDICTOR == "boltz":
    CONFIRMED = False        # <-- set True only AFTER reading the printed estimate

    !pip install -q boltz-api 2>/dev/null || pip install -q boltz_api
    # device-code flow works headless; follow the printed URL
    !boltz-api auth login --device-code

    import subprocess, yaml, numpy as np
    WORK = "/content/boltz_jobs"; os.makedirs(WORK, exist_ok=True)

    def payload(name, seq):
        return {"entities": [
                    {"type":"protein","chain_ids":["A"],"value": TARGET_SEQ},
                    {"type":"protein","chain_ids":["B"],"value": seq}],
                "binding": {"type":"protein_protein_binding","binder_chain_ids":["B"]},
                "num_samples": 5}

    total = 0.0
    for n in todo:
        p = f"{WORK}/{n}.yaml"; yaml.safe_dump(payload(n, designs[n]), open(p,"w"))
        r = subprocess.run(["boltz-api","predictions:structure-and-binding","estimate-cost",
                            "--model","boltz-2.1","--input",f"@yaml://{p}"],
                           capture_output=True, text=True)
        print(f"{n:<28} {r.stdout.strip()[:120]}")
    print(f"\n{len(todo)} jobs to submit. Read the per-job estimates above and sum them.")

    if not CONFIRMED:
        print("\n>>> STOPPED. Nothing has been billed.")
        print(">>> Set CONFIRMED = True and re-run this cell to submit.")
    else:
        for i, n in enumerate(todo, 1):
            p = f"{WORK}/{n}.yaml"
            r = subprocess.run(["boltz-api","predictions:structure-and-binding","start",
                                "--model","boltz-2.1","--idempotency-key",f"degfp-{n}",
                                "--input",f"@yaml://{p}","--raw-output","--transform","id"],
                               capture_output=True, text=True)
            jid = r.stdout.strip()
            if not jid: print(f"[{i}/{len(todo)}] {n} submit failed: {r.stderr[-300:]}"); continue
            ckpt[key("boltz", n)] = {"boltz_job_id": jid, "status": "submitted"}; save()
            print(f"[{i}/{len(todo)}] {n:<28} job {jid}")
        print("\nSubmitted. Run the next cell to download results once they finish.")
else:
    print(f"skipped (PREDICTOR={PREDICTOR})")


### 4b · Download Boltz results and extract ipSAE

Re-runnable. Skips anything already downloaded.

In [ ]:
if PREDICTOR == "boltz":
    import subprocess, tarfile, numpy as np
    ROOT = f"{DRIVE_ROOT}/boltz-runs"; os.makedirs(ROOT, exist_ok=True)
    for n in designs:
        k = key("boltz", n)
        if k not in ckpt or ckpt[k].get("ipsae_median") is not None: continue
        jid = ckpt[k]["boltz_job_id"]
        subprocess.run(["boltz-api","download-results","--id",jid,"--name",n,
                        "--root-dir",ROOT,"--poll-interval-seconds","10"])
        mets = glob.glob(f"{ROOT}/{n}/**/metrics.json", recursive=True)
        if not mets:
            for tp in glob.glob(f"{ROOT}/{n}/**/archive.tar.gz", recursive=True):
                tarfile.open(tp).extractall(os.path.dirname(tp))
            mets = glob.glob(f"{ROOT}/{n}/**/metrics.json", recursive=True)
        vals = []
        for m in mets:
            j = json.load(open(m))
            for kk in ("ipsae_min","ipsae","min_ipsae","iptm"):
                if kk in j: vals.append(float(j[kk])); break
        if vals:
            ckpt[k].update({"ipsae_median": float(np.median(vals)),
                            "ipsae_spread": float(np.max(vals)-np.min(vals)),
                            "n_samples": len(vals), "status": "done"})
            save(); print(f"{n:<28} ipSAE {np.median(vals):.3f}  spread {np.max(vals)-np.min(vals):.3f}")
        else:
            print(f"{n:<28} no metrics found yet - job may still be running")
else:
    print(f"skipped (PREDICTOR={PREDICTOR})")


## 5 · Merge and rank by consensus

Run this after all three predictors have been through. Ranks by **rank-sum** across the metrics —
the scheme that put the one real DWV binder first by a 7-point margin.

In [ ]:
import pandas as pd, numpy as np
ckpt = json.load(open(CKPT))
rows = {}
for k, v in ckpt.items():
    pred, name = k.split("::", 1)
    rows.setdefault(name, {}).update(v)
df = pd.DataFrame(rows).T
if df.empty:
    print("checkpoint is empty"); raise SystemExit

SPEC = [("ipsae_median", False), ("protenix_iptm", False),
        ("esm_complex_ptm", False), ("esm_monomer_plddt", False),
        ("protenix_seed_spread", True)]          # (column, lower_is_better)
have = [(c, lo) for c, lo in SPEC if c in df.columns and df[c].notna().any()]
print("ranking on:", [c for c, _ in have])
if len(have) < 2:
    print("!! fewer than 2 metrics present - run more predictors before trusting this")

for c, lo in have:
    df[f"rk_{c}"] = df[c].rank(ascending=lo, na_option="bottom")
df["ranksum"] = df[[f"rk_{c}" for c, _ in have]].sum(axis=1)
df = df.sort_values("ranksum")

show = [c for c, _ in have] + ["ranksum"]
display(df[show].round(3))

out = f"{DRIVE_ROOT}/consensus_ranked.csv"
df.to_csv(out)
print(f"\nwrote {out}")
print("\nTop of this table is what to order. Sanity-check the leader's seed spread before you commit:")
print("a high spread means the model gave a different answer each run and you only looked at one.")


## 6 · Status — what has been scored so far

In [ ]:
import json, os, collections
ckpt = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
done = collections.defaultdict(set)
for k in ckpt:
    pred, name = k.split("::", 1); done[pred].add(name)
print(f"checkpoint: {CKPT}  ({len(ckpt)} entries, {os.path.getsize(CKPT)/1e3:.1f} KB)" if ckpt else "checkpoint empty")
for pred in ("esmfold", "protenix", "boltz"):
    missing = [n for n in designs if n not in done[pred]]
    print(f"  {pred:<10} {len(done[pred]):>3} done, {len(missing):>3} remaining"
          + (f"   next: {missing[0]}" if missing else "   COMPLETE"))
print("\nNothing here is lost on disconnect. Re-run the cell for whichever predictor")
print("is incomplete and it picks up exactly where it stopped.")
